# PARAKH — Stage 8 training round

Two models. **Neither predicts fraud**, because no fraud label exists.

| model | task | ground truth | claim |
|---|---|---|---|
| Consistency NN | mask a field, predict it from the rest | the field's own value | *this record disagrees with itself* |
| Work-text encoder | description → agency / cost band / status | the record's own columns | *these two works are peers* |

Both are feature work. Neither moves the calibration gate — see `artifacts/stage8/APPROVAL_STATUS.json`.

**Attach dataset `parakh-corpus` containing:**
`synthetic_dataset.csv`, `ground_truth_ledger.json`, `parakh_consistency_train.py`, `parakh_nlp_train.py`


In [ ]:
import os, json, glob, shutil, sys
from pathlib import Path

SRC = None
for c in glob.glob('/kaggle/input/*'):
    if list(Path(c).glob('parakh_*_train.py')):
        SRC = Path(c); break
assert SRC, 'Attach the parakh-corpus dataset (must contain the two *_train.py files)'

WORK = Path('/kaggle/working')
for f in SRC.glob('parakh_*_train.py'):
    shutil.copy(f, WORK / f.name)

CSV = next(iter(SRC.glob('synthetic_dataset.csv')), None)
LEDGER = next(iter(SRC.glob('ground_truth_ledger.json')), None)
print('scripts :', [f.name for f in WORK.glob('parakh_*_train.py')])
print('corpus  :', CSV)
print('ledger  :', LEDGER)
print('GPU     :', bool(os.environ.get('CUDA_VISIBLE_DEVICES')) or 'check Settings > Accelerator')


---
## Round 1 — Consistency NN  *(CPU is fine, ~10 min)*

Local baseline to beat, demo data, seed 20260906:

| channel | n | ROC-AUC |
|---|---|---|
| cost_outlier | 171 | **0.699** |
| date_order | 142 | **0.608** |
| overspend | 113 | 0.596 |
| agency_mismatch | 113 | 0.548 |

⚠ **Reconstruction accuracy is not the success metric.** An earlier version
scored 0.97 reconstruction and 0.538 defect detection. Only the second is the job.

⚠ Ignore the `state` head — districts nest inside states, so it is a lookup table.


In [ ]:
!cd /kaggle/working && python parakh_consistency_train.py --demo --epochs 60 --out /kaggle/working/consistency_demo


In [ ]:
# On the real corpus (skips automatically if the CSV was not attached)
if CSV:
    !cd /kaggle/working && python parakh_consistency_train.py --csv {CSV} --epochs 60 --out /kaggle/working/consistency_real
else:
    print('no synthetic_dataset.csv attached; demo run only')


In [ ]:
m = json.load(open('/kaggle/working/consistency_demo/metrics.json'))
print('epochs run:', m['manifest']['epochs_run'], '| early stopped:', m['manifest']['early_stopped'])
print()
print('reconstruction (diagnostic only)')
for k, v in m['reconstruction'].items():
    print(f"  {k:22s} {v['accuracy']:.4f}  base {v['mode_baseline']:.4f}  lift {v['lift']:+.4f}")
print()
print('DEFECT DETECTION  <- this is the result')
BASE = {'cost_outlier':0.699,'date_order':0.608,'overspend':0.596,'agency_mismatch':0.548}
for k, v in m['defect_detection'].items():
    auc = v['roc_auc_vs_clean']; ref = BASE.get(k)
    tag = '' if ref is None else ('BEAT' if auc > ref else 'below')
    print(f"  {k:22s} n={v['n_positive']:5d}  AUC {auc:.4f}  {tag} {ref if ref else ''}")


---
## Round 2 — gate first, then train  *(GPU T4 x2, internet ON for first run)*

The probe runs **before** any training. It embeds `road`/`sadak`,
`school`/`vidyalaya`, `building`/`bhavan` against unrelated controls.

```
separation > 0.05  →  continue
separation ≤ 0.05  →  STOP, do not train heads
```

Cross-lingual synonymy is the *entire* reason to replace TF-IDF. If the encoder
does not have it, training heads burns GPU for a number that cannot justify the swap.


In [ ]:
!pip -q install transformers


In [ ]:
# GATE — 30 seconds, decides the whole round
!cd /kaggle/working && python -c "
from parakh_nlp_train import synonymy_probe, ENCODERS
import json
p = synonymy_probe(ENCODERS['muril'])
print(json.dumps({k:v for k,v in p.items() if k!='per_pair'}, indent=2))
print()
for pair, sim in p['per_pair'].items(): print(f'  {pair:28s} {sim:.4f}')
print()
print('PASS - continue' if p['separation'] > 0.05 else 'FAIL - stop here')
"


In [ ]:
# Only run this if the gate printed PASS
!cd /kaggle/working && python parakh_nlp_train.py --encoder muril --demo --epochs 30 --skip-probe --out /kaggle/working/nlp_muril


In [ ]:
n = json.load(open('/kaggle/working/nlp_muril/metrics.json'))
print('encoder:', n['encoder_model'])
print()
print('target                 neural   tfidf+lr   winner')
for k, v in n['targets'].items():
    print(f"  {k:20s} {v['neural_accuracy']:.4f}   {v['tfidf_lr_accuracy']:.4f}   {v['winner']}")
print()
print(n['verdict'])


---
## Collect

Commit the notebook, then download `/kaggle/working/*/metrics.json`.
Those are what the Stage 8 report cites.


In [ ]:
for f in sorted(glob.glob('/kaggle/working/*/metrics.json')):
    print('='*70); print(f)
    print(json.dumps(json.load(open(f)), indent=2)[:1500])


---
### Reminders

- `defect_channel` describes **injected data-quality faults** in a synthetic
  corpus. Valid as a validation target here; never a fraud label.
- Keep MuRIL **frozen**. 236M parameters will memorise 20k four-word phrases.
- Neither model's output is a fraud score. Neither has seen a fraud label.
